# 3_0_2 对话检索（G 段）

**前置 Notebook**

| 步骤 | Notebook | 内容 |
|------|----------|------|
| KG 构建 | `1_2_0_2build_kg__neo4j.ipynb` | 知识图谱 |
| 索引与 MetaPath | `1_2_1_2pagerankMetapath.ipynb` | PageRank、embedding/fulltext、mid/low |

**本 Notebook**：多轮 **LangGraph** 检索 + 答案生成（论文 §5.1 / §5.4）。不包含 Neo4jSchemaManager（v3 已移除，检索不依赖 LLM 动态 Cypher）。


## Part 0 · 架构导读

### LangGraph 六节点（与 `build_graph_rag_pipeline` 一致）

```
改写(N1) → 路由(N2) → 候选池(N3) → 召回(N4) → 重排(N5) → 答案(N6)
```

| LangGraph 节点 | 论文章节 | 中文职责 |
|----------------|----------|----------|
| `query_rewriter` | — | 问题改写成英文检索短语 |
| `route` | §5.1 | 选主题模块 r、路径层级 l、多轮操作 κ |
| `gsub_builder` | §5.1 | 构造 OLAP 软先验池 `gsub_mp_ids` |
| `recall` | §5.2 | Hybrid 广召回 Top-50 |
| `rerank` | §5.2 | 融合检索分/图分/OLAP 先验 → Top-10 |
| `answer_generator` | §5.4 | 路径级 Context + 答案 + 写回状态 |

### 准备层 / 评估层（非 LangGraph 节点）

| 编号 | 内容 |
|------|------|
| **P0–P3** | 依赖、状态、Neo4j/LLM 连接 |
| **Prep** | 图算子 import、Recall 用 Cypher 模板 |
| **E1–E5** | 指标说明、评估脚本、OLAP 对比 |


## 术语表（读代码与评估前先看）

| 中文 | 字段 / 符号 | 说明 |
|------|-------------|------|
| 主题模块 | `target_subgraphs` (r) | MPU / EEM / EBM，不是 mid/low |
| 路径层级 | `path_level` (l) | mid 概览，low 细节；首轮由 Route/LLM 选择 |
| 多轮操作 | `kappa` (κ) | first_turn / drill_down / roll_up / sibling_nav / drill_across |
| 上一轮短名单 | `candidate_mp_ids` (C) | 上轮 P*，约 10 条 |
| OLAP 先验池 | `gsub_mp_ids` | Turn2+ 图算子展开；**不**限制 Recall 范围 |
| 广召回池 | `recall_candidates` | Recall 输出，最多 50 条 |
| 本轮短名单 | `retrieval_mp_ids` (P*) | Rerank 后 Top-10 |


## 数据流与流程图

```mermaid
flowchart LR
    Q[用户问题] --> N1[改写]
    N1 --> N2[路由 r,l,κ]
    N2 --> N3[建池 G_sub]
    N3 --> N4[Recall Top-50]
    N4 --> N5[Rerank Top-10]
    N5 --> N6[答案]
    N6 --> M[写回 M_t]
```

**实验开关**：`PIPELINE_VARIANT = "full" | "no_hierarchy"`（见 P2 Code Cell）。


### P0 环境与依赖导入

**在整体中的位置**：Notebook 最前；为 P2–P3 与全部 Node 提供路径与 `dotenv`。

**本 Code Cell 做什么**

- 加载 `PaperExtract/output` 等路径（历史字段，v3 检索不读动态 Schema 文件）
- 导入 `langgraph`、`neo4j_graphrag` 等包

**关键输出**：无 graph state；仅副作用导入。


In [38]:
# Cell 1: 导入依赖
import os
import re
import json
from pathlib import Path
from typing import TypedDict, Annotated, Dict, List, Union, Optional, Tuple
import operator

from dotenv import load_dotenv
from neo4j import GraphDatabase

from neo4j_graphrag.embeddings.sentence_transformers import SentenceTransformerEmbeddings
from neo4j_graphrag.llm import OpenAILLM
from neo4j_graphrag.retrievers import HybridCypherRetriever

from langgraph.graph import StateGraph, END

# 加载环境变量
load_dotenv()

print("✅ 依赖导入成功")

# 验证环境变量
required_vars = ["NEO4J_URI", "NEO4J_USER", "NEO4J_PASSWORD", 
                 "LOCAL_MODEL_PATH_BCE", "SCHEMA_PATH"]
missing_vars = [var for var in required_vars if var not in os.environ]

if missing_vars:
    print(f"⚠️  缺少环境变量: {missing_vars}")
else:
    print("✅ 所有环境变量已加载")
    print(f"Schema路径: {os.environ['SCHEMA_PATH']}")

✅ 依赖导入成功
✅ 所有环境变量已加载
Schema路径: C:/Users/tom/OneDrive/LUCK/luck grpahrag/code/PaperExtract/output


### P2 对话状态 M_t 与实验开关

**在整体中的位置**：Pipeline 运行前定义；`make_initial_state()` 供首轮与评估调用。

**本 Code Cell 做什么**

- 定义 `SimplifiedGraphRAGState`（TypedDict）
- 设置 `PIPELINE_VARIANT`：`full` | `no_hierarchy`

**关键字段**

| 字段 | 含义 |
|------|------|
| `target_subgraphs` | 主题模块 r |
| `path_level` | l = mid / low |
| `kappa` | 多轮操作 κ |
| `candidate_mp_ids` | 上一轮 P* |
| `gsub_mp_ids` | OLAP 先验池（Node 3 写入） |
| `recall_candidates` | Recall Top-50（Node 4 写入） |
| `retrieval_mp_ids` | 本轮 P* Top-10（Node 5 写入） |

**`no_hierarchy`**：Turn≥2 强制 `κ=first_turn, l=mid`（消融 Stateful 导航，见 `pipeline_config.py`）。

**`no_hierarchy`**（Stateful 消融）：Turn≥2 仅强制 `κ=first_turn`；**l 仍由 Route LLM 选择**（见 `pipeline_config.py`）。

**`session_mode`**：`stateful`（默认）| `stateless`（E5 扁平检索基线，见 Part 2 对照表）。



In [ ]:
# ══════════════════════════════════════════════════════════════
# G Cell 2: State 定义（论文 §5.1 对话状态 M_t）
# ══════════════════════════════════════════════════════════════

from __future__ import annotations

from typing import TypedDict, List, Annotated
import operator

from utilities.pipeline_config import set_pipeline_variant, get_pipeline_config

# ── 实验开关：修改此处切换 full / no_hierarchy ──
PIPELINE_VARIANT = "full"
set_pipeline_variant(PIPELINE_VARIANT)

VALID_KAPPA = frozenset({
    "first_turn", "drill_down", "roll_up", "sibling_nav", "drill_across",
})
VALID_PATH_LEVELS = frozenset({"mid", "low"})
TOP_LEVEL_MODULES = frozenset({"MPU", "EEM", "EBM"})


class SimplifiedGraphRAGState(TypedDict):
    """
    对话状态 M_t = ⟨r_t, C_t, l_t⟩ + κ + 检索/答案字段。

    r_t  : target_subgraphs   — 顶层语义模块 {EBM,EEM,MPU}（非 path_level）
    C_t  : candidate_mp_ids   — 上一轮排序后候选 MetaPath 集合 P*
    l_t  : path_level         — 路径抽象层级 mid | low
    κ    : kappa                — 结构转移类型
    """

    original_query: str
    rewritten_query: str
    final_answer: str

    schema_info: str
    keywords_zh: list[str]
    keywords_en: list[str]
    keywords_both: list[str]

    # r — 顶层语义模块（实现字段名 target_subgraphs / subgraph）
    target_subgraphs: list[str]

    # M_t 核心
    path_level: str
    kappa: str
    candidate_mp_ids: list[str]   # C_{t-1} / C_t = P*
    anchor_mp_ids: list[str]      # 与 C 同步，兼容旧字段
    entity_ids: list[str]         # E_t = ⋃ V_p
    dialogue_turn: int
    session_mode: str  # stateful | stateless（E5 对照）
    previous_query: str  # q_{n-1}，Route_olap 必填（Turn≥2）

    # G_sub（Route_olap 之后；仅 Rerank 偏置，不限制 Recall）
    gsub_mp_ids: list[str]
    gsub_size: int
    # 检索 / 答案
    recall_candidates: list  # Node4 C_rec
    recall_count: int
    retrieval_mp_ids: list[str]   # 本轮 P* (Top-10)
    retrieval_results: str

    generated_cypher: str
    cypher_valid: bool
    retry_count: int
    error_log: list[str]


def make_initial_state(
    query: str, *, session_mode: str = "stateful"
) -> SimplifiedGraphRAGState:
    """初始化单轮 state。Stateless 评估每轮调用且 session_mode='stateless'。"""
    from utilities.session_config import normalize_session_mode

    mode = normalize_session_mode(session_mode)
    return {
        "original_query": query,
        "rewritten_query": "",
        "previous_query": "",
        "final_answer": "",
        "schema_info": "",
        "keywords_zh": [],
        "keywords_en": [],
        "keywords_both": [],
        "target_subgraphs": [],
        "path_level": "mid",
        "kappa": "first_turn",
        "candidate_mp_ids": [],
        "anchor_mp_ids": [],
        "entity_ids": [],
        "dialogue_turn": 0,
        "session_mode": mode,
        "gsub_mp_ids": [],
        "gsub_size": 0,
        "recall_candidates": [],
        "recall_count": 0,
        "retrieval_mp_ids": [],
        "retrieval_results": "",
        "generated_cypher": "",
        "cypher_valid": False,
        "retry_count": 0,
        "error_log": [],
    }


print("✅ SimplifiedGraphRAGState（§5.1 M_t）")
print("   r=target_subgraphs (顶层 EBM/EEM/MPU)")
print("   l=path_level (mid/low), C=candidate_mp_ids")
print(f"   PIPELINE_VARIANT={get_pipeline_config().variant} "
      f"flags={get_pipeline_config().feature_flags()}")


### P3 模型与 Neo4j 连接

**在整体中的位置**：所有 Node 之前执行；提供全局 `llm`、`neo4j_embed_model`、`neo4j_driver`。

**本 Code Cell 做什么**

- `SentenceTransformerEmbeddings`（本地向量模型）
- `OpenAILLM`（改写 / 路由 / 答案 / 评估 judge）
- `GraphDatabase.driver` 连接 Neo4j

**关键环境变量**：`NEO4J_URI`、`NEO4J_USER`、`NEO4J_PASSWORD`、`QWEN_API_KEY`、`LOCAL_MODEL_PATH_BCE` 等（见 `.env`）。

**运行注意**：必须先本 Cell 再跑 Pipeline；否则 `neo4j_driver` / `llm` 未定义。


In [41]:
# ============================================================
# Agentic Graph RAG vs Native Graph RAG — 对比测试
#
# 新增功能：
#   1. 原生 Graph RAG（无迭代，单次检索生成）
#   2. Agentic Graph RAG（多轮迭代优化）
#   3. 并行执行对比
#   4. 结果评估和统计
# ============================================================

import os
import re
import time
from typing import TypedDict, Annotated, Dict, List
import operator

from dotenv import load_dotenv
from neo4j import GraphDatabase

from neo4j_graphrag.embeddings.sentence_transformers import SentenceTransformerEmbeddings
from neo4j_graphrag.llm import OpenAILLM
from neo4j_graphrag.retrievers import HybridCypherRetriever

from langgraph.graph import StateGraph, END

# ══════════════════════════════════════════════════════════════
# 1. 环境与模型初始化
# ══════════════════════════════════════════════════════════════
load_dotenv()
os.environ["TRANSFORMERS_OFFLINE"] = "1"
os.environ["HF_DATASETS_OFFLINE"] = "1"

# 向量模型
neo4j_embed_model = SentenceTransformerEmbeddings(
    model=os.environ["LOCAL_MODEL_PATH_BCE"]
)

# LLM
#llm = OpenAILLM(
#    model_name="qwen/qwen3.5-9b",
#    base_url="http://127.0.0.1:1234/v1",
#    api_key="not-needed",
#    model_params={"temperature": 0.7, "max_tokens": 1024},
#)
# 初始化 LLM
llm = OpenAILLM(
    model_name="qwen-plus",
    base_url="https://dashscope.aliyuncs.com/compatible-mode/v1",
    api_key=os.environ.get("QWEN_API_KEY"),  # ← 修正：不要用列表
    model_params={
        "temperature": 0.1,
        "max_tokens": 1024,
    },
)

# 测试调用
try:
    response = llm.invoke("你好，请说'测试成功'")
    print("✅ 模型可用")
    print(f"响应: {response.content}")
except Exception as e:
    print("❌ 模型不可用")
    print(f"错误: {e}")
#llm=OpenAILLM(
#                    model_name="deepseek-chat",
#                    model_params={
#                        "max_tokens": 8000,
#                        "temperature": 0.3,
#                        "top_p": 0.9,
#                        "frequency_penalty": 0.1,
#                    },
#                    api_key=os.environ["DEEPSEEK_API_KEY"],
#                    base_url='https://api.deepseek.com/v1'
 #               )
# Neo4j



neo4j_driver = GraphDatabase.driver(
    os.environ["NEO4J_URI"],
    auth=(os.environ["NEO4J_USER"], os.environ["NEO4J_PASSWORD"]),
)
print("✅ Neo4j连接成功")


✅ 模型可用
响应: 测试成功
✅ Neo4j连接成功


### N1 查询改写（Query Rewriter）

**在整体中的位置**：LangGraph 入口 → 输出 `rewritten_query` → Node 2 Route。

**本 Code Cell 做什么**

- 中文：关键词表 + LLM 英译改写
- 英文：LLM 压缩为 ≤20 词检索短语
- 改写失败或过短：**raise**（无固定 fallback 短语）

**关键参数**：改写长度上限约 120 字符；**不使用** Neo4j Schema。

**输入 / 输出**

| 方向 | 字段 |
|------|------|
| 入 | `original_query` |
| 出 | `rewritten_query`，`keywords_*`（`schema_info` 保留字段但恒为空） |


In [42]:
# ══════════════════════════════════════════════════════════════
# N1: Query Rewriter
# ══════════════════════════════════════════════════════════════

from typing import Dict, List


def extract_and_translate_keywords(query: str) -> dict:
    """中文领域关键词 → 英文（词典映射，无 LLM）。"""
    keyword_map = {
        "汞": ["mercury", "Hg", "HgT", "MeHg"],
        "铅": ["lead", "Pb"],
        "镉": ["cadmium", "Cd"],
        "砷": ["arsenic", "As"],
        "铬": ["chromium", "Cr"],
        "重金属": ["heavy metal", "heavy metals"],
        "水稻": ["rice", "paddy rice", "Oryza"],
        "大米": ["rice", "rice grain"],
        "籽粒": ["grain", "seed"],
        "样本": ["sample", "specimen"],
        "标本": ["specimen"],
        "检测": ["detection", "determination", "measurement"],
        "分析": ["analysis", "assay"],
        "方法": ["method", "approach", "technique"],
        "实验": ["experiment", "test"],
        "测定": ["determination", "measurement"],
        "消解": ["digestion", "dissolution"],
        "数据": ["data"],
        "数据集": ["dataset", "data set"],
        "结果": ["result", "outcome"],
        "结论": ["conclusion", "finding"],
        "污染": ["contamination", "pollution"],
        "浓度": ["concentration", "level"],
        "含量": ["content", "amount"],
        "采集": ["collection", "sampling"],
        "保存": ["preservation", "storage"],
    }
    chinese_kws = []
    english_kws = []
    for cn_word, en_words in keyword_map.items():
        if cn_word in query:
            chinese_kws.append(cn_word)
            english_kws.extend(en_words)
    english_kws = list(dict.fromkeys(english_kws))
    both = []
    for cn_word in chinese_kws:
        en_words = keyword_map.get(cn_word, [])
        if en_words:
            both.append(cn_word + "|" + "|".join(en_words[:2]))
    return {
        "chinese_keywords": chinese_kws,
        "english_keywords": english_kws,
        "both": both,
        "has_keywords": len(chinese_kws) > 0,
    }


def query_rewriter_node(state: SimplifiedGraphRAGState) -> Dict:
    print("\n" + "=" * 60)
    print("Node 1: Query Rewriter")
    print("=" * 60)

    original = state["original_query"]
    if not original or not original.strip():
        raise ValueError("original_query 为空")

    print(f"原问题: {original}")
    keywords = extract_and_translate_keywords(original)
    if keywords["has_keywords"]:
        print(f"  [关键词] 中文: {keywords['chinese_keywords']}")
        print(f"  [关键词] 英文: {keywords['english_keywords'][:5]}")

    is_chinese = _is_chinese_query(original)
    print(f"  [语言] {'中文' if is_chinese else '英文'}")

    if is_chinese:
        prompt = _build_translate_and_rewrite_prompt(original, keywords)
    else:
        prompt = _build_english_rewrite_prompt(original)

    print("  [改写] 调用 LLM...")
    rewritten = llm.invoke(prompt).content.strip()
    rewritten = _clean_rewritten_query(rewritten)

    if not rewritten or len(rewritten) < 3:
        raise ValueError(
            f"改写结果过短或为空: {rewritten!r}; keywords={keywords['english_keywords'][:5]}"
        )

    if len(rewritten) > 120:
        print(f"  ⚠ 改写过长({len(rewritten)}字符)，词边界截断")
        rewritten = rewritten[:120].rsplit(" ", 1)[0]

    print(f"改写后(英文): {rewritten}")

    return {
        "rewritten_query": rewritten,
        "schema_info": "",
        "keywords_zh": keywords["chinese_keywords"],
        "keywords_en": keywords["english_keywords"],
        "keywords_both": keywords["both"],
    }


def _is_chinese_query(query: str) -> bool:
    chinese_chars = sum(1 for c in query if "\u4e00" <= c <= "\u9fff")
    return chinese_chars / max(len(query.strip()), 1) > 0.3


def _build_translate_and_rewrite_prompt(query: str, keywords: Dict) -> str:
    keyword_hint = ""
    if keywords["english_keywords"]:
        en_kws = keywords["english_keywords"][:5]
        keyword_hint = f"Key terms: {', '.join(en_kws)}\n"
    return f"""You are a scientific literature search expert. Translate the Chinese query to a concise English search phrase.

Chinese query: {query}
{keyword_hint}
Instructions:
- Output English ONLY, maximum 20 words
- Preserve the EXACT semantic intent, do NOT expand or add context
- Output a noun phrase or short declarative sentence
- NEVER use: whu_, mp_, iao_, prov_ prefixes

Output (20 words max):"""


def _build_english_rewrite_prompt(query: str) -> str:
    return f"""You are a scientific literature search expert. Rewrite the query as a concise search phrase.

Query: {query}

Instructions:
- Output English ONLY, maximum 20 words
- Preserve the EXACT semantic intent, do NOT expand or add new concepts
- Output a noun phrase or short declarative sentence
- NEVER use: whu_, mp_, iao_, prov_ prefixes

Output (20 words max):"""


def _clean_rewritten_query(query: str) -> str:
    prefixes = [
        "Rewritten query:", "Query:", "English:",
        "Translated:", "Output:", "Result:",
        "改写后", "翻译", "查询",
    ]
    for prefix in prefixes:
        if query.lower().startswith(prefix.lower()):
            query = query[len(prefix) :].strip()
            break
    query = query.strip("\"'\"\"''")
    return query.rstrip(".").strip()


print("✅ N1 query_rewriter_node 定义完成")


✅ Node 1 辅助函数定义完成
  extract_and_translate_keywords : 关键词提取（197个词条映射）
  _get_schema_summary_from_manager: Schema 摘要
  _get_full_schema_for_cypher     : 完整 Schema
✅ Node 1 定义完成（纯英文改写版）


### N2a Route_r（Recall 之前）

仅 LLM 选择 **r**（MPU/EEM/EBM）。Recall 在模块内 **flat** 检索。


In [ ]:
# ══════════════════════════════════════════════════════════════
# Node 2a: Route_r — Recall 前仅选顶层模块 r
# ══════════════════════════════════════════════════════════════

from utilities.dialogue_routing import route_modules_recall, route_modules_only
from utilities.session_config import is_stateless, PATH_LEVEL_FLAT


def route_r_node(state: SimplifiedGraphRAGState) -> Dict:
    print("\n" + "=" * 60)
    print("Node 2a: Route_r (modules only)")
    print("=" * 60)

    query = state.get("rewritten_query") or state["original_query"]
    if not query or not query.strip():
        raise ValueError("Route_r 需要非空查询")

    print(f"查询: {query[:80]}...")
    print(f"  turn={state.get('dialogue_turn', 0)} session={state.get('session_mode')}")

    if is_stateless(state):
        modules = route_modules_only(llm, query)
        print(f"  → [stateless] r={modules} (κ/l 在 Route 节点占位)")
        return {
            "target_subgraphs": modules,
            "kappa": "first_turn",
            "path_level": PATH_LEVEL_FLAT,
        }

    modules = route_modules_recall(llm, query)
    print(f"  → r={modules} (Recall 将 module-flat 检索，不按 l 过滤)")
    return {"target_subgraphs": modules}


print("✅ Node 2a route_r_node 定义完成")


## Part 2 · 检索与重排（Recall + Rerank）

### 设计原则（Stateful · Search Wide, Rank Narrow）

| 阶段 | 范围 |
|------|------|
| **Recall** | 模块图全集（mid+low flat），**不**按 κ / G_sub / l 过滤 |
| **Route_olap + G_sub** | 结构导航，仅影响 **Rerank** |
| **Rerank** | $s_{\mathrm{final}} = \alpha s_{\mathrm{search}} + \eta s_{\mathrm{pr}} + \gamma s_{\mathrm{olap}}$ |

### 重排公式（Node 5 · Stateful）

$$s_{\mathrm{final}} = \alpha s_{\mathrm{search}} + \eta s_{\mathrm{pr}} + \gamma s_{\mathrm{olap}}$$

| 符号 | 代码 | Stateful 默认 | 含义 |
|------|------|---------------|------|
| $s_{\mathrm{search}}$ | hybrid `score` | 归一化 | 语义相关性 |
| $s_{\mathrm{pr}}$ | `maxPageRank` | 归一化 | 图分析分；缺失 **raise** |
| $s_{\mathrm{olap}}$ | G_sub 内/外 | **1.0 / 0.0** | 结构对齐（二元，无 0.3 软兜底） |
| α, η, γ | `RerankWeights` | 0.5, 0.35, 0.15 | Turn1: γ=0；Turn2+ κ≠first_turn: γ=0.15 且 **G_sub 非空** |

实现：`utilities/retrieval_rerank.py` → `rerank_metapath_candidates()`。

### Stateful vs Stateless（E5 OLAP 对照）

| 维度 | Stateful (`session_mode=stateful`) | Stateless (`session_mode=stateless`) |
|------|-----------------------------------|--------------------------------------|
| 轮间状态 | 保留 C、r、l、κ、`dialogue_turn`、`previous_query` | 每轮 `make_initial_state`，**无记忆** |
| Route | **Route_r**（r）→ Recall → **Route_olap**（κ,l）→ G_sub | **仅 r**；κ=`first_turn` |
| G_sub | Turn2+ κ 展开；**空池 raise** | **恒空** |
| Recall | **模块级 flat**（与 Stateless 同宽） | **flat** |
| Rerank | α·检索 + η·PR + γ·OLAP | **仅 η·PR**（α=0，γ=0） |
| 改写 / 答案 | 保留 | 保留 |

**E5 读数**：Turn1 两臂 Recall 机制趋同；Turn2+ 看 OLAP 是否通过 **Rerank** 提升 Precision 且不牺牲 Recall。


### Prep 图算子与重排函数导入

**在整体中的位置**：Node 3/4/5 之前执行一次；仅 import，无 state 更新。

**本 Code Cell 做什么**

- `N_l`, `WF`, `DA`, `build_gsub_mp_ids`（`dialogue_routing.py`）
- `RECALL_TOP_K=50`, `OUTPUT_TOP_K=10`, `rerank_metapath_candidates`

**关键常量**：广召回 50 条，最终 P* 10 条。


In [ ]:
from utilities.retrieval_rerank import (
    RECALL_TOP_K,
    OUTPUT_TOP_K,
    rerank_metapath_candidates,
    rerank_page_rank_only,
)
from utilities.session_config import is_stateless, PATH_LEVEL_FLAT
from utilities.recall_flat import (
    build_cypher_for_subgraph_flat,
    HYBRID_SCAN_FLAT_PER_MODULE,
)
# ══════════════════════════════════════════════════════════════
# G Cell 4: G_sub 算子 + Rerank 导入
# ══════════════════════════════════════════════════════════════

from utilities.dialogue_routing import (
    SIBLING_EDGE_TYPES,
    TOP_LEVEL_MODULES,
    VALID_KAPPA,
    VALID_PATH_LEVELS,
    N_l,
    WF,
    DA,
    build_gsub_mp_ids,
)

print("✅ G_sub 算子 + rerank 已导入")
print(f"   RECALL_TOP_K={RECALL_TOP_K}, OUTPUT_TOP_K={OUTPUT_TOP_K}")
print(f"   顶层模块: {sorted(TOP_LEVEL_MODULES)}")


### N3 G_sub（Route_olap 之后）

**在整体中的位置**：Recall **之后**；由 κ 展开 `gsub_mp_ids`，**仅**供 Rerank。

**严格模式**：Turn≥2 且 κ≠first_turn 时 G_sub 为空 → **RuntimeError**（不 warn 继续）。


In [ ]:
# ══════════════════════════════════════════════════════════════
# Node 3: G_sub Builder（Route_olap 之后；空池 Turn≥2 raise）
# ══════════════════════════════════════════════════════════════

from utilities.pipeline_config import get_pipeline_config
from utilities.session_config import is_stateless


def gsub_builder_node(state: SimplifiedGraphRAGState) -> Dict:
    print("\n" + "=" * 60)
    print("Node 3: G_sub Builder")
    print("=" * 60)

    kappa = state["kappa"]
    path_level = state["path_level"]
    modules = state["target_subgraphs"]
    candidates = state.get("candidate_mp_ids") or []

    print(f"  κ={kappa}, l={path_level}, r={modules}")
    print(f"  |C_{{t-1}}|={len(candidates)}")

    cfg = get_pipeline_config()
    if is_stateless(state):
        print("  G_sub = ∅ (stateless)")
        return {"gsub_mp_ids": [], "gsub_size": 0}

    if not cfg.gsub_enabled:
        if kappa != "first_turn":
            raise RuntimeError(
                f"variant={cfg.variant} 禁用 G_sub，但 κ={kappa}"
            )
        print("  G_sub = ∅ (variant 禁用 / first_turn)")
        return {"gsub_mp_ids": [], "gsub_size": 0}

    if kappa == "first_turn":
        print("  G_sub = ∅ (首轮；Rerank γ=0)")
        return {"gsub_mp_ids": [], "gsub_size": 0}

    gsub_ids = build_gsub_mp_ids(
        driver=neo4j_driver,
        kappa=kappa,
        candidate_mp_ids=candidates,
        active_modules=modules,
        path_level=path_level,
    )

    if not gsub_ids:
        raise RuntimeError(
            f"G_sub 为空: κ={kappa}, l={path_level}, |C|={len(candidates)} — "
            "禁止静默降级；请检查图算子或 Route_olap 标注"
        )

    print(f"  G_sub = {len(gsub_ids)} 条 MetaPath（Rerank OLAP 偏置）")
    return {"gsub_mp_ids": gsub_ids, "gsub_size": len(gsub_ids)}


print("✅ Node 3 gsub_builder_node 定义完成")


### N4 广召回 Recall（Hybrid Top-50）

**在整体中的位置**：按 Route 的 r、l 在各模块做 hybrid；**与 κ 无关**（每轮均广召回）。

**本 Code Cell 做什么**

- `HybridCypherRetriever` + `HYBRID_SCAN_TOP_K`（mid 300 / low 60）
- 合并去重 → 最多 **50** 条 → `recall_candidates`

**输入 / 输出**

| 入 | `rewritten_query`, `target_subgraphs`, `path_level` |
| 出 | `recall_candidates`, `recall_count` |


### Prep Recall 用 Cypher 模板

**在整体中的位置**：Node 4 Recall 调用 `_build_cypher_for_subgraph(sg, l)`。

**本 Code Cell 做什么**

- 定义 hybrid 检索后的 Cypher 投影（Chunk 文本、`maxPageRank` 等）
- `graph_score = node.maxPageRank`（**无 COALESCE**）

**关键参数**：`subgraph` ∈ MPU/EEM/EBM；`path_level` ∈ mid/low。


In [ ]:
# ══════════════════════════════════════════════════════════════
# G Cell 6: MetaPath 检索 Cypher（Recall）
# ══════════════════════════════════════════════════════════════


def _build_cypher_for_subgraph(subgraph: str, path_level: str = "mid") -> str:
    if path_level not in VALID_PATH_LEVELS:
        raise ValueError(f"无效 path_level: {path_level}")
    if subgraph not in TOP_LEVEL_MODULES:
        raise ValueError(f"无效顶层模块: {subgraph}")
    return f"""
WITH node
WHERE node.subgraph = '{subgraph}' AND node.path_level = '{path_level}'
OPTIONAL MATCH (node)-[r:metaPathRelation]->(entity)-[:FROM_CHUNK]->(chunk:Chunk)
WITH node, r.position AS position, chunk
ORDER BY position ASC
WITH node, COLLECT(chunk.text) AS chunk_texts_ordered
WITH node,
     reduce(acc = [], x IN chunk_texts_ordered |
            CASE WHEN x IN acc OR x IS NULL OR size(x) <= 10
                 THEN acc ELSE acc + x END) AS chunk_texts
RETURN
    node.metaPathText AS metapath_text,
    chunk_texts AS chunk_texts,
    node.maxPageRank AS graph_score,
    node.mp_id AS mp_id,
    node.path_level AS path_level,
    node.subgraph AS subgraph,
    node.path_type AS path_type,
    node.metaPathQuery AS meta_path_query
"""


print("✅ MetaPath Recall Cypher 定义完成（graph_score = maxPageRank，无 COALESCE）")


In [ ]:
# ══════════════════════════════════════════════════════════════
# Node 4: Recall（hybrid Top-50，全 κ 统一广召回）
# ══════════════════════════════════════════════════════════════

from neo4j_graphrag.retrievers import HybridCypherRetriever
from neo4j_graphrag.types import RetrieverResultItem
from utilities.pipeline_config import get_pipeline_config
from utilities.session_config import is_stateless
from utilities.recall_flat import (
    build_cypher_for_subgraph_flat,
    HYBRID_SCAN_FLAT_PER_MODULE,
)
import neo4j
import re
from typing import Dict, List, Any

METAPATH_VECTOR_INDEX = "metapath_embedding_index"
METAPATH_FULLTEXT_INDEX = "metapath_fulltext_index"
HYBRID_SCAN_TOP_K = {"mid": 300, "low": 60}


def sanitize_for_lucene(text: str) -> str:
    return re.sub(r'[+\-&|!(){}\[\]^"~*?:\\/—–]', ' ', text).strip()


def metapath_result_formatter(record: neo4j.Record) -> RetrieverResultItem:
    return RetrieverResultItem(content=dict(record), metadata=None)


def _search_single_subgraph(
    query_text: str, subgraph: str, path_level: str
) -> List[Dict]:
    retrieval_query = _build_cypher_for_subgraph(subgraph, path_level)
    retriever = HybridCypherRetriever(
        driver=neo4j_driver,
        vector_index_name=METAPATH_VECTOR_INDEX,
        fulltext_index_name=METAPATH_FULLTEXT_INDEX,
        embedder=neo4j_embed_model,
        retrieval_query=retrieval_query,
        result_formatter=metapath_result_formatter,
    )
    scan_k = HYBRID_SCAN_TOP_K.get(path_level, RECALL_TOP_K)
    retriever_result = retriever.search(query_text=query_text, top_k=scan_k)
    results = _safe_convert_results(retriever_result)
    for r in results:
        r["_subgraph"] = subgraph
    return results


def _search_single_subgraph_flat(query_text: str, subgraph: str) -> List[Dict]:
    retrieval_query = build_cypher_for_subgraph_flat(subgraph)
    retriever = HybridCypherRetriever(
        driver=neo4j_driver,
        vector_index_name=METAPATH_VECTOR_INDEX,
        fulltext_index_name=METAPATH_FULLTEXT_INDEX,
        embedder=neo4j_embed_model,
        retrieval_query=retrieval_query,
        result_formatter=metapath_result_formatter,
    )
    retriever_result = retriever.search(
        query_text=query_text, top_k=HYBRID_SCAN_FLAT_PER_MODULE
    )
    results = _safe_convert_results(retriever_result)
    for r in results:
        r["_subgraph"] = subgraph
    return results


def recall_node(state: SimplifiedGraphRAGState) -> Dict:
    print("\n" + "=" * 60)
    print("Node 4: Recall (hybrid Top-50)")
    print("=" * 60)

    query_text = sanitize_for_lucene(state["rewritten_query"])
    if not query_text:
        raise ValueError("检索查询为空")

    modules = state["target_subgraphs"]
    if not modules:
        raise ValueError("target_subgraphs 为空")

    if is_stateless(state):
        print(f"查询: {query_text[:80]}...")
        print(f"r={modules} | κ=first_turn | Recall=flat (无 l 过滤)")
        all_results: List[Dict] = []
        for sg in modules:
            print(f"\n  ── hybrid {sg} (flat) ──")
            batch = _search_single_subgraph_flat(query_text, sg)
            print(f"  返回: {len(batch)}")
            all_results.extend(batch)
        if not all_results:
            raise RuntimeError(f"Stateless Recall 无结果: r={modules}")
        all_results.sort(key=lambda x: float(x.get("score") or 0.0), reverse=True)
        merged = _deduplicate_by_mp_id(all_results)[:RECALL_TOP_K]
        print(f"  C_rec: {len(merged)} 条 (cap={RECALL_TOP_K})")
        return {"recall_candidates": merged, "recall_count": len(merged)}

    cfg = get_pipeline_config()
    if not cfg.multi_dim_enabled:
        pass  # Recall 已统一 flat；κ/l 仅用于 Route_olap / G_sub / Rerank

    print(f"查询: {query_text[:80]}...")
    print(f"r={modules} | Stateful Recall=flat (module-wide, no l filter)")

    all_results: List[Dict] = []
    for sg in modules:
        print(f"\n  ── hybrid {sg} (flat) ──")
        batch = _search_single_subgraph_flat(query_text, sg)
        print(f"  返回: {len(batch)}")
        all_results.extend(batch)

    if not all_results:
        raise RuntimeError(f"Stateful Recall 无结果: r={modules}")

    all_results.sort(key=lambda x: float(x.get("score") or 0.0), reverse=True)
    merged = _deduplicate_by_mp_id(all_results)[:RECALL_TOP_K]

    print(f"  C_rec: {len(merged)} 条 (cap={RECALL_TOP_K})")

    return {
        "recall_candidates": merged,
        "recall_count": len(merged),
    }


def _deduplicate_by_mp_id(results: List[Dict]) -> List[Dict]:
    seen: set = set()
    merged: List[Dict] = []
    for item in results:
        mp_id = item.get("mp_id")
        if mp_id is None:
            merged.append(item)
        elif mp_id not in seen:
            seen.add(mp_id)
            merged.append(item)
    return merged


def _safe_convert_results(retriever_result: Any) -> List[Dict]:
    items = (
        retriever_result.items
        if hasattr(retriever_result, "items")
        else (
            retriever_result
            if isinstance(retriever_result, (list, tuple))
            else [retriever_result]
        )
    )
    results = []
    for item in items:
        converted = _convert_single_item(item)
        if converted:
            results.append(converted)
    return results


def _convert_single_item(item: Any) -> Dict:
    if isinstance(item, dict):
        return item
    if hasattr(item, "data") and callable(getattr(item, "data", None)):
        try:
            return dict(item.data())
        except Exception:
            pass
    if hasattr(item, "content"):
        content = item.content
        if isinstance(content, dict):
            return content
        if isinstance(content, str) and content.startswith("<Record"):
            return {"raw_content": content}
        if content is not None:
            try:
                return dict(content)
            except Exception:
                return {"raw_content": str(content)}
    if hasattr(item, "metadata") and isinstance(item.metadata, dict):
        return item.metadata
    raise TypeError(f"无法转换检索结果项: {type(item)}")


print("✅ Node 4 recall_node 定义完成")


### N2b Route_olap（Recall 之后）

**输入**：`previous_query` + 当前 query + `candidate_mp_ids`。
**输出**：`kappa`, `path_level`（**不**改 `target_subgraphs`）。


In [ ]:
# ══════════════════════════════════════════════════════════════
# Node 2b: Route_olap — Recall 后 (q_{n-1}, q_n) → κ, l
# ══════════════════════════════════════════════════════════════

from utilities.dialogue_routing import route_olap_dialogue
from utilities.pipeline_config import get_pipeline_config
from utilities.session_config import is_stateless, PATH_LEVEL_FLAT


def route_olap_node(state: SimplifiedGraphRAGState) -> Dict:
    print("\n" + "=" * 60)
    print("Node 2b: Route_olap (κ, l)")
    print("=" * 60)

    if is_stateless(state):
        print("  [stateless] 跳过 Route_olap")
        return {}

    query = state.get("rewritten_query") or state["original_query"]
    if not query.strip():
        raise ValueError("Route_olap 需要非空查询")

    prev_q = state.get("previous_query") or ""
    turn = int(state.get("dialogue_turn") or 0)
    print(f"  q_n: {query[:60]}...")
    if turn > 0:
        print(f"  q_{{n-1}}: {prev_q[:60]}...")

    cfg = get_pipeline_config()
    routed = route_olap_dialogue(
        llm, query, prev_q, state, allowed_kappa=cfg.allowed_olap_modes,
    )
    routed = cfg.apply_olap_route_override(
        routed,
        dialogue_turn=turn,
        llm=llm,
        query=query,
    )
    print(f"  → κ={routed['kappa']}, l={routed['path_level']} (r 保持 {state.get('target_subgraphs')})")
    return {
        "kappa": routed["kappa"],
        "path_level": routed["path_level"],
    }


print("✅ Node 2b route_olap_node 定义完成")


### N5 重排序 Rerank（Top-10）

**在整体中的位置**：读 `recall_candidates` + `gsub_mp_ids` → 输出 P*。

**本 Code Cell 做什么**

- 调用 `rerank_metapath_candidates`（见 Part 2 公式说明）
- `κ=first_turn` → **γ=0**；否则 γ=0.15

**输入 / 输出**

| 出 | `retrieval_mp_ids`, `candidate_mp_ids`（10 条） |


In [ ]:
# ══════════════════════════════════════════════════════════════
# Node 5: Rerank（α·检索 + η·PR + γ·OLAP → Top-10）
# ══════════════════════════════════════════════════════════════

import json
from typing import Dict, List, Any
from utilities.session_config import is_stateless


def _format_retrieval_results(results: List[Dict]) -> Dict:
    return {
        "status": "success",
        "count": len(results),
        "results": [
            {
                "rank": i + 1,
                "mp_id": item.get("mp_id"),
                "combined_score": item.get("combined_score"),
                "s_search": item.get("s_search"),
                "s_pr": item.get("s_pr"),
                "s_olap": item.get("s_olap"),
                "content": item,
                "preview": _generate_preview(item),
                "subgraph": item.get("_subgraph") or item.get("subgraph", "unknown"),
            }
            for i, item in enumerate(results)
        ],
    }


def _generate_preview(item: Dict) -> str:
    mp_text = (item.get("metapath_text") or "").strip()
    chunk_texts = item.get("chunk_texts") or []
    if mp_text:
        ctx = ""
        if isinstance(chunk_texts, list):
            parts = [str(c).strip()[:150] for c in chunk_texts[:2] if c]
            ctx = " | ".join(parts)
        return mp_text[:250] + (f"\n[Context] {ctx}" if ctx else "")
    raise ValueError(f"preview 缺少 metapath_text: mp_id={item.get('mp_id')}")


def rerank_node(state: SimplifiedGraphRAGState) -> Dict:
    print("\n" + "=" * 60)
    print("Node 5: Rerank (Top-10)")
    print("=" * 60)

    query_text = sanitize_for_lucene(state["rewritten_query"])
    if not query_text:
        raise ValueError("Rerank 需要非空 rewritten_query")

    pool = state.get("recall_candidates") or []
    if not pool:
        raise ValueError("rerank_node: recall_candidates 为空")

    if is_stateless(state):
        print(f"  |C_rec|={len(pool)} | stateless rerank: η·PageRank only")
        ranked = rerank_page_rank_only(pool)
        top = ranked[:OUTPUT_TOP_K]
    else:
        kappa = state["kappa"]
        path_level = state["path_level"]
        gsub_ids = state.get("gsub_mp_ids") or []
        gamma = 0.0 if kappa == "first_turn" else None

        print(
            f"  |C_rec|={len(pool)} |G_sub|={len(gsub_ids)} | κ={kappa} | "
            f"γ={gamma if gamma is not None else 'default'}"
        )

        ranked = rerank_metapath_candidates(
            neo4j_embed_model,
            query_text,
            pool,
            gsub_mp_ids=gsub_ids,
            gamma=gamma if gamma is not None else 0.15,
        )
        top = ranked[:OUTPUT_TOP_K]

    p_star = [r["mp_id"] for r in top if r.get("mp_id")]

    if len(p_star) != len(top):
        raise RuntimeError("Rerank 结果存在缺失 mp_id 的行")

    formatted = _format_retrieval_results(top)
    print(f"  P* Top-{OUTPUT_TOP_K}: {p_star[:3]}...")

    return {
        "retrieval_results": json.dumps(formatted, ensure_ascii=False, indent=2),
        "retrieval_mp_ids": p_star,
        "candidate_mp_ids": p_star,
        "anchor_mp_ids": p_star[:5],
        "path_level": state["path_level"],
        "kappa": state["kappa"],
    }


print("✅ Node 5 rerank_node 定义完成")


### N6 证据上下文与答案（§5.4）

**在整体中的位置**：Pipeline 出口；写回多轮状态。

**本 Code Cell 做什么**

- 对 P* 前 10 条构建 `Context(p)`（路径结构 + 有序 Chunk）
- LLM 生成带引用答案
- 写回 `candidate_mp_ids`、`dialogue_turn+1` 等

**输入 / 输出**

| 入 | `original_query`, `retrieval_mp_ids` |
| 出 | `final_answer`，更新后的 M_t |


In [ ]:
# ══════════════════════════════════════════════════════════════
# Node 6: §5.4 Evidence Context + Answer + 状态写回
# ══════════════════════════════════════════════════════════════

from typing import Dict, List
import json

from utilities.dialogue_routing import (
    build_context_for_paths,
    extract_entity_ids,
)


def answer_generator_node(state: SimplifiedGraphRAGState) -> Dict:
    print("\n" + "=" * 60)
    print("Node 6: Answer Generator (§5.4)")
    print("=" * 60)

    original_query = state["original_query"]
    modules = state.get("target_subgraphs") or []
    p_star = state.get("retrieval_mp_ids") or state.get("candidate_mp_ids") or []

    if not p_star:
        raise ValueError("answer_generator: 缺少 retrieval_mp_ids (P*)")

    print(f"原问题: {original_query}")
    print(f"r={modules} | |P*|={len(p_star)}")

    context = build_context_for_paths(neo4j_driver, p_star, max_paths=10)
    print(f"  Context(q) 长度: {len(context)} 字符")

    prompt = _build_answer_prompt(
        question=original_query,
        context=context,
        subgraphs=modules,
    )
    answer = _generate_answer_with_llm(prompt)
    entity_ids = extract_entity_ids(neo4j_driver, p_star)

    print(f"  |E_t|={len(entity_ids)}  ✅ 答案生成完成")

    return {
        "final_answer": answer,
        "target_subgraphs": modules,
        "candidate_mp_ids": p_star,
        "anchor_mp_ids": p_star[:5],
        "retrieval_mp_ids": p_star,
        "entity_ids": entity_ids,
        "path_level": state["path_level"],
        "kappa": state["kappa"],
        "dialogue_turn": int(state.get("dialogue_turn") or 0) + 1,
    }


def _build_answer_prompt(question: str, context: str, subgraphs: List[str]) -> str:
    hints = {
        "MPU": "论证与证据（声明、数据集、结论）",
        "EEM": "实验与方法（步骤、仪器、质控）",
        "EBM": "样本与材料（采集、环境、浓度）",
    }
    hint = "；".join(hints[s] for s in subgraphs if s in hints) or "综合各模块"
    return f"""你是科研助手。基于路径级证据 Context(q) 用中文回答。

用户问题：{question}

Context(q) 每条含 [T_struct]（实体-关系骨架）与 [OrderedChunks]（按路径顺序的原文）：
{context}

要求：
1. 重点维度：{hint}
2. 关键陈述标注 [编号]，对应 Context 中 [N]
3. 不足处明确说明缺失信息
4. 2-3 段，专业准确

答案："""


def _generate_answer_with_llm(prompt: str) -> str:
    response = llm.invoke(prompt)
    answer = response.content.strip() if hasattr(response, "content") else str(response).strip()
    if len(answer) < 10:
        raise RuntimeError("LLM 返回过短答案")
    return answer


print("✅ Node 5 (§5.4 + 状态写回) 定义完成")


### 组装 LangGraph Pipeline

**在整体中的位置**：定义 `graph_app`；评估与多轮测试均 `graph_app.invoke(state)`。

**本 Code Cell 做什么**

- `build_graph_rag_pipeline()` 注册六节点与边
- 可选 `test_multiturn_dialogue()` 冒烟

**运行顺序（手动）**：P0 → P2 → P3 → Prep import → N1–N6 定义 → 本 Cell → 评估 Cell。


In [1]:
# ══════════════════════════════════════════════════════════════
# G Cell 7: Pipeline + 多轮测试（6 Node）
# ══════════════════════════════════════════════════════════════

from __future__ import annotations

from typing import List, Dict
from langgraph.graph import StateGraph, END
import json
import time


def build_graph_rag_pipeline():
    workflow = StateGraph(SimplifiedGraphRAGState)

    workflow.add_node("query_rewriter", query_rewriter_node)
    workflow.add_node("route_r", route_r_node)
    workflow.add_node("recall", recall_node)
    workflow.add_node("route_olap", route_olap_node)
    workflow.add_node("gsub_builder", gsub_builder_node)
    workflow.add_node("rerank", rerank_node)
    workflow.add_node("answer_generator", answer_generator_node)

    workflow.set_entry_point("query_rewriter")
    workflow.add_edge("query_rewriter", "route_r")
    workflow.add_edge("route_r", "recall")
    workflow.add_edge("recall", "route_olap")
    workflow.add_edge("route_olap", "gsub_builder")
    workflow.add_edge("gsub_builder", "rerank")
    workflow.add_edge("rerank", "answer_generator")
    workflow.add_edge("answer_generator", END)
    return workflow.compile()


def invoke_dialogue_turn(state: SimplifiedGraphRAGState) -> SimplifiedGraphRAGState:
    """单轮 invoke；多轮时传入上轮 state 并更新 original_query / previous_query。"""
    return graph_app.invoke(state)


def test_multiturn_dialogue():
    print("\n" + "=" * 80)
    print("多轮对话测试 (Route_r → Recall → Route_olap → G_sub → Rerank)")
    print("=" * 80)

    turns = [
        ("大米中汞污染的整体情况", "first_turn"),
        ("展开具体检测步骤细节", "drill_down / low"),
        ("回到概览层面", "roll_up / mid"),
    ]

    state = make_initial_state(turns[0][0])
    prev_query = turns[0][0]
    for i, (query, label) in enumerate(turns, 1):
        print(f"\n── Turn {i}: {label} ──")
        print(f"Q: {query}")
        if i > 1:
            state["previous_query"] = prev_query
            state["original_query"] = query
            state["rewritten_query"] = ""
        state = invoke_dialogue_turn(state)
        prev_query = query
        print(
            f"  κ={state.get('kappa')} l={state.get('path_level')} "
            f"r={state.get('target_subgraphs')} |G_sub|={len(state.get('gsub_mp_ids') or [])} "
            f"|P*|={len(state.get('candidate_mp_ids') or [])}"
        )

    print("\n✅ 多轮测试完成")


graph_app = build_graph_rag_pipeline()

print("✅ Pipeline: Rewriter → Route_r → Recall(flat) → Route_olap → G_sub → Rerank → Answer")

try:
    from IPython.display import Image, display
    display(Image(graph_app.get_graph().draw_mermaid_png()))
except Exception as exc:
    print(f"流程图渲染失败: {exc}")


NameError: name 'SimplifiedGraphRAGState' is not defined

## Part 4 · 评估

### E1 评估指标说明

| 指标 | 含义 | 备注 |
|------|------|------|
| Recall@10 | 相关路径被召回到 Top-10 的比例 | 无 qrels 时用 LLM judge（proxy） |
| Precision@10 | Top-10 中相关路径占比 | 同上 |
| anchor_overlap | Turn2 的 P* 与 Turn1 P* 的 mp_id 交集 | 衡量多轮锚定 |
| faithfulness | 答案是否可由 Context 支持 | LLM judge |
| answer_relevance | 答案与问题相关性 | LLM judge |
| context_precision | 检索上下文相关性 | LLM judge |

数据：`data/dialogue_test_cases.json`（多轮）；`data/questions.csv`（单轮 legacy）。


### E2 评估模块导入

**本 Code Cell**：导入 `utilities/test_evaluation`（`evaluate_dialogue_scenarios`、`evaluate_olap_comparison` 等）。

**输出日志**：`output/eval_log.md`（追加写入）。


In [ ]:
# ══════════════════════════════════════════════════════════════
# 评估共享模块（OLAP 核心指标 + 路由命中率）
# ══════════════════════════════════════════════════════════════

from utilities.test_evaluation import (
    resolve_questions_csv,
    load_test_cases,
    load_dialogue_test_cases,
    load_dialogue_test_cases_with_meta,
    evaluate_test_cases,
    evaluate_dialogue_scenarios,
    evaluate_olap_comparison,
    compare_ablation_reports,
    run_dialogue_scenario,
    run_stateless_scenario,
    print_evaluation_summary,
    run_single_case,
    append_eval_log,
    format_eval_log_md,
    CORE_METRIC_KEYS,
)

QUESTIONS_CSV = str(resolve_questions_csv())
print(f"✅ 测试 CSV: {QUESTIONS_CSV}")
print(f"✅ OLAP 核心指标: {list(CORE_METRIC_KEYS)}")


### E3 Stateful 多轮对话测试

对 `dialogue_test_cases.json` 跑 **Stateful** 全流程并打核心指标（默认 `PIPELINE_VARIANT=full`）。


In [ ]:
# ══════════════════════════════════════════════════════════════
# 多轮对话测试 + OLAP 核心指标（Stateful, qrels优先/LLM fallback）
# ══════════════════════════════════════════════════════════════

from utilities.dialogue_test_set import format_test_set_banner
from utilities.olap_modes import configure_olap_modes, format_olap_modes_banner, get_active_olap_modes
from utilities.pipeline_config import get_pipeline_config

# 测试集：legacy10 | all | first（见 dialogue_test_set.py）
DIALOGUE_TEST_SET = "legacy10"
# OLAP 实验 κ：core = first_turn+drill_down+roll_up；extended；all
OLAP_MODES = "core"
configure_olap_modes(OLAP_MODES)

_scenarios, _ts_meta = load_dialogue_test_cases_with_meta(
    test_set=DIALOGUE_TEST_SET, olap_modes=OLAP_MODES,
)
print(format_test_set_banner(_ts_meta))
print(format_olap_modes_banner(get_active_olap_modes()))
print(f"Pipeline variant: {get_pipeline_config().variant}")

dialogue_report = evaluate_dialogue_scenarios(
    graph_app,
    make_initial_state,
    _scenarios,
    embedder=neo4j_embed_model,
    llm=llm,
    neo4j_driver=neo4j_driver,
    session_mode="stateful",
    score_core_metrics=True,
    verbose=True,
)
print("\nTurn2 核心指标摘要:")
_summary = dialogue_report["summary"]
for k in CORE_METRIC_KEYS:
    key = f"turn2_avg_{k}"
    if key in _summary:
        print(f"  {k}: {_summary.get(key)}")


### E4 单轮 / 综合测试（可选）

`questions.csv` 单轮路由与 legacy 指标；按需运行，非 OLAP 主实验。


In [ ]:
# ══════════════════════════════════════════════════════════════
# E4: 单轮综合测试（questions.csv，可选）
# ══════════════════════════════════════════════════════════════

test_cases = load_test_cases(QUESTIONS_CSV)
for case in test_cases:
    print(f"  {case['query'][:42]:<42} → {case['expected']}")

report = evaluate_test_cases(
    graph_app,
    make_initial_state,
    test_cases,
    embedder=neo4j_embed_model,
    verbose=True,
)
print_evaluation_summary(report)


### E5 OLAP 对比（Stateful vs Stateless）

**成对运行**同一批 scenario（测试集由 `DIALOGUE_TEST_SET` / 下方 cell 控制，见 `utilities/dialogue_test_set.py`）：

- `legacy10`（默认）：严格 q01–q10，与历史 v3 同集对比  
- `all`：JSON 全量 66 条  
- `first`：JSON 前 N 条（≠ legacy10，扩展集后勿与 legacy10 混用）


| 臂 | 机制 |
|----|------|
| **Stateful** | 6 节点 + 多轮记忆 + Turn2+ κ / G_sub / γ |
| **Stateless** | 每轮独立 state；Route **只选 r**；Recall **不区分 mid/low**；Rerank **仅 PageRank** |

**报告结构**

1. **Turn1 sanity**：Stateful − Stateless，Δ 应接近 0  
2. **Turn2 core Δ**：Precision@10、Recall@10、anchor_overlap、faithfulness 等  

CLI：`python utilities/run_retrieval_eval.py --olap-compare --skip-comprehensive`

结果追加写入 `output/eval_log.md`。



In [ ]:
# ══════════════════════════════════════════════════════════════
# OLAP 对比评估：Stateful vs Stateless + 写 eval_log.md
# ══════════════════════════════════════════════════════════════

from datetime import datetime, timezone
from utilities.dialogue_test_set import format_test_set_banner
from utilities.olap_modes import configure_olap_modes, format_olap_modes_banner, get_active_olap_modes
from utilities.pipeline_config import PipelineConfig

DIALOGUE_TEST_SET = "all"  # all=全 JSON；legacy10=q01–q10
OLAP_MODES = "core"  # first_turn + drill_down + roll_up
configure_olap_modes(OLAP_MODES)

_scenarios, _ts_meta = load_dialogue_test_cases_with_meta(
    test_set=DIALOGUE_TEST_SET, olap_modes=OLAP_MODES,
)
print(format_test_set_banner(_ts_meta))
print(format_olap_modes_banner(get_active_olap_modes()))
_started = datetime.now(timezone.utc).strftime("%Y-%m-%d %H:%M:%S UTC")

report_stateful = evaluate_dialogue_scenarios(
    graph_app, make_initial_state, _scenarios,
    embedder=neo4j_embed_model, llm=llm, neo4j_driver=neo4j_driver,
    session_mode="stateful", score_core_metrics=True, verbose=True,
    dialogue_test_set=_ts_meta.get("test_set"),
    olap_modes=_ts_meta.get("olap_modes"),
)
report_stateless = evaluate_dialogue_scenarios(
    graph_app, make_initial_state, _scenarios,
    embedder=neo4j_embed_model, llm=llm, neo4j_driver=neo4j_driver,
    session_mode="stateless", score_core_metrics=True, verbose=True,
    dialogue_test_set=_ts_meta.get("test_set"),
    olap_modes=_ts_meta.get("olap_modes"),
)
olap = evaluate_olap_comparison(report_stateful, report_stateless)

print("\n" + "=" * 72)
print("Turn1 Sanity (Stateful − Stateless, 应 ≈ 0)")
print("=" * 72)
for key, vals in olap["turn1_sanity"].items():
    print(f"  {key}: Δ={vals.get('delta')}")

print("\nTurn2 Core Δ (Stateful − Stateless)")
for key, vals in olap["turn2_delta"].items():
    print(f"  {key}: stateful={vals.get('stateful')} stateless={vals.get('stateless')} Δ={vals.get('delta')}")

_flags = PipelineConfig(variant=PIPELINE_VARIANT).feature_flags()
_md = format_eval_log_md(
    meta={
        "timestamp": _started + " [notebook olap]",
        "version": "notebook-olap-compare",
        "dialogue_test_set": _ts_meta.get("test_set"),
        "pipeline_variant": PIPELINE_VARIANT,
        "session_mode": "paired",
        "retrieval_scoring": "qrels优先 / 无则 LLM judge",
        "feature_flags": _flags,
        "exit": "success",
    },
    dialogue_report=report_stateful,
    olap_compare=olap,
)
_log = append_eval_log(_md)
print(f"\n📝 eval log: {_log}")
